<a href="https://colab.research.google.com/github/gitly-br/PSA_pred_models/blob/modelos_v2/notebooks/Processamento_chamados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução

# Preparação

## Imports

In [51]:
import pandas as pd
import os
import copy
import json

## Funções

Leitura de arquivos. Primeiro tenta achar localmente, depois busca pelo link do drive.

In [52]:
def load_csv_dataframe(file_name, drive_link=None):
    local_path = file_name
    if os.path.exists(local_path):
        print(f"'{file_name}' found locally. Loading...")
        df = pd.read_csv(local_path)
        return df.copy()
    elif drive_link:
        print(f"'{file_name}' not found locally. Attempting to load from Google Drive link...")
        try:
            # Attempt to convert a standard Google Drive share link to a direct download link
            if "drive.google.com" in drive_link and "view" in drive_link:
                file_id = drive_link.split('/d/')[-1].split('/view')[0].split('?')[0]
                direct_download_link = f"https://drive.google.com/uc?export=download&id={file_id}"
                print(f"Converted Google Drive link to direct download: {direct_download_link}")
                df = pd.read_csv(direct_download_link)
            else:
                df = pd.read_csv(drive_link) # Assume it's already a direct readable URL

            print(f"Successfully loaded from Google Drive link.")
            return copy.deepcopy(df)
        except Exception as e:
            print(f"Error loading from Google Drive link: {e}")
            raise FileNotFoundError(
                f"'{file_name}' not found locally and could not be loaded from Google Drive link."
            )
    else:
        raise FileNotFoundError(
            f"'{file_name}' not found locally and no Google Drive link provided."
        )

def load_json_to_dict(file_name, drive_link=None):
    local_path = file_name
    if os.path.exists(local_path):
        print(f"'{file_name}' found locally. Loading...")
        with open(local_path, 'r') as f:
            data = json.load(f)
        return copy.deepcopy(data)
    elif drive_link:
        print(f"'{file_name}' not found locally. Attempting to load from Google Drive link...")
        try:
            import requests
            # Attempt to convert a standard Google Drive share link to a direct download link
            if "drive.google.com" in drive_link and "view" in drive_link:
                file_id = drive_link.split('/d/')[-1].split('/view')[0].split('?')[0]
                direct_download_link = f"https://drive.google.com/uc?export=download&id={file_id}"
                print(f"Converted Google Drive link to direct download: {direct_download_link}")
                response = requests.get(direct_download_link)
            else:
                response = requests.get(drive_link)

            response.raise_for_status() # Raise an exception for HTTP errors
            data = json.loads(response.text)
            print(f"Successfully loaded from Google Drive link.")
            return copy.deepcopy(data)
        except requests.exceptions.RequestException as e:
            print(f"Error fetching from Google Drive link: {e}")
            raise FileNotFoundError(
                f"'{file_name}' not found locally and could not be loaded from Google Drive link (Request Error)."
            )
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from Google Drive link: {e}")
            raise ValueError(
                f"Could not decode JSON from Google Drive link for '{file_name}'."
            )
        except Exception as e:
            print(f"An unexpected error occurred loading from Google Drive link: {e}")
            raise FileNotFoundError(
                f"'{file_name}' not found locally and could not be loaded from Google Drive link."
            )
    else:
        raise FileNotFoundError(
            f"'{file_name}' not found locally and no Google Drive link provided."
        )


## Datasets externos

Chamados sem processamento algum

In [64]:
df_chamados_raw = load_csv_dataframe(
    "/content/chamados_raw.csv",
    drive_link="https://drive.google.com/file/d/1f5Dlf0DAwDZa8GoBJKRpf0TBFjUuzdAP/view?usp=drive_link"
    )

'/content/chamados_raw.csv' not found locally. Attempting to load from Google Drive link...
Converted Google Drive link to direct download: https://drive.google.com/uc?export=download&id=1f5Dlf0DAwDZa8GoBJKRpf0TBFjUuzdAP
Successfully loaded from Google Drive link.


Definição de bairros por bacia

In [68]:
bacias_bairros = load_json_to_dict(
    "/content/bacias.json",
    drive_link="https://drive.google.com/file/d/15oplqaHWY20ad7aL8Dt7AMvDFe_uTtq9/view?usp=drive_link"
)

'/content/bacias.json' not found locally. Attempting to load from Google Drive link...
Converted Google Drive link to direct download: https://drive.google.com/uc?export=download&id=15oplqaHWY20ad7aL8Dt7AMvDFe_uTtq9
Successfully loaded from Google Drive link.


# Separação por bairros

Extraindo bairro do endereço

In [79]:
import re

neighborhood_pattern = r'.* -\s*(.+)$'

df_chamados = df_chamados_raw.copy()
df_chamados['bairro'] = df_chamados['end'].astype(str).str.extract(neighborhood_pattern, flags=re.IGNORECASE)[0]
df_chamados['bairro'] = df_chamados['bairro'].str.strip()

Resolvendo problemas no registro de bairro de chamados

In [80]:
import unicodedata

# 1. Define abbreviation_map
abbreviation_map = {
    'JARDIM': 'JD',
    'VILA': 'VL',
    'PARQUE': 'PQ',
    'CONJUNTO RESIDENCIAL': 'CJ RES',
    'CIDADE': 'CD',
    'SETOR': 'ST',
    'DISTRITO INDUSTRIAL': 'DIST IND',
    'NUCLEO HABITACIONAL': 'NUC HAB',
    'RESIDENCIAL': 'RES'
}

# 2. Define specific_corrections_map
specific_corrections_map = {
    'VARZEA DO TAMANDUATE': 'VARZEA DO TAMANDUATEI',
    'VL FRANCISCO MATARAZ': 'VL FRANCISCO MATARAZZO',
    'PQ GERASSI CENTREVIL': 'PQ GERASSI',
    'JARDIM CLUBE DE CAMP': 'JD CLUBE DE CAMPO', # Correcting truncated name
    'ESTANCIA DO RIO GRAN': 'ESTANCIA DO RIO GRANDE', # Correcting truncated name
    'RECREIO DA BORDA DO': 'RECREIO DA BORDA DO CAMPO', # Correcting truncated name
    'ACAMPAMENTO ANCHIETA': 'ACAMPAMENTO ANCHIETA', # Keep as is, it's a specific area name
    'ASS. ESPIRITO SANTO, 117': 'JD ESPIRITO SANTO', # Assuming this refers to a neighborhood
    'BAIRRO INEXISTENTE': 'BAIRRO INEXISTENTE', # Keep as is, if it's truly an unknown bairro
    'CAMPO GRANDE': 'CAMPO GRANDE', # Keep as is
    'JARDIM DO MIRANTE': 'JD DO MIRANTE', # Standardizing
    'JARDIM SANTO ANDRÉ': 'JD SANTO ANDRE', # Standardizing
    'PARANAPIACABA': 'PARANAPIACABA', # Keep as is
    'RIO GRANDE': 'RIO GRANDE', # Keep as is
    'SITIO TAQUARAL': 'SITIO TAQUARAL', # Keep as is
    'TAMANDUATEÍ 2': 'TAMANDUATEI 2', # Standardizing accent
    'TAMANDUATEÍ 3': 'TAMANDUATEI 3', # Standardizing accent
    'TAMANDUATEÍ 8': 'TAMANDUATEI 8', # Standardizing accent
    'VARZEA DO TAMANDUATEI': 'VARZEA DO TAMANDUATEI', # Already corrected or correct
    'VILA JOÃO RAMALHO': 'VL JOAO RAMALHO', # Standardizing
    'VL FRANCISCO MATARAZZO': 'VL FRANCISCO MATARAZZO' # Already corrected or correct
}

# 3. Define function to remove accents and convert to uppercase
def remove_accents(text):
    if pd.isna(text):
        return text
    text = str(text).upper()
    nfkd_form = unicodedata.normalize('NFKD', text)
    only_ascii = nfkd_form.encode('ascii', 'ignore').decode('utf-8')
    return only_ascii.strip()

# 4. Apply remove_accents function
df_chamados['bairro'] = df_chamados['bairro'].apply(remove_accents)

# 5. Apply abbreviation_map
for full_form, abbr in abbreviation_map.items():
    df_chamados['bairro'] = df_chamados['bairro'].str.replace(full_form, abbr, regex=False)

# 6. Apply specific_corrections_map
for misspelled, corrected in specific_corrections_map.items():
    df_chamados['bairro'] = df_chamados['bairro'].str.replace(misspelled, corrected, regex=False)


print("Bairro standardization complete. Displaying unique values after transformation:")
print(df_chamados['bairro'].unique())

Bairro standardization complete. Displaying unique values after transformation:
['VL VITORIA' 'RECREIO DA BORDA DO CAMPO' 'VL ALZIRA' 'VL BASTOS'
 'VL METALURGICA' 'JD RIVIERA' 'VL PALMARES' 'JD SANTO ANDRE' 'VL LUZITA'
 'CENTRO' 'PQ DAS NACOES' 'JD SANTA CRISTINA' 'VL LINDA' 'SANTA MARIA'
 'VL GUIOMAR' 'VL FLORESTA' 'CONDOMINIO MARACANA' 'VL JOAO RAMALHO'
 'CATA PRETA' 'VL GUARANI' 'JD SANTO ANTONIO' 'VL SACADURA CABRAL'
 'VL HOMERO THON' 'VL CURUCA' 'PQ CAPUAVA' 'JD UTINGA' 'VL ASSUNCAO'
 'VL GILDA' 'JD IPANEMA' 'VL HUMAITA' 'VL AMERICA' 'SILVEIRA' 'JD VL RICA'
 'JD TELLES DE MENEZES' 'PQ JOAO RAMALHO' 'PARAISO' 'JD' 'CAMPESTRE'
 'VL MARINA' 'VL SCARPELLI' 'VL SUICA' 'CD SAO JORGE' 'JD CRISTIANE'
 'VL PRINCIPE DE GALES' 'SANTA TEREZINHA' 'JD BOM PASTOR' 'VL LUCINDA'
 'VL HELENA' 'VL GUARACIABA' 'PQ ERASMO ASSUNCAO' 'JD ALZIRA FRANCO'
 'BANGU' 'PQ ORATORIO' 'PQ NOVO ORATORIO' 'JD STELLA' 'JD MILENA'
 'JD BELA VISTA' 'VL VALPARAISO' 'VL CAMILOPOLIS' 'JD ITAPOAN' 'JD IRENE'
 'VL PROGRES

Checando bairros que existem nos chamados mas não pertencem a nenhuma bacia

In [81]:
# Get all unique neighborhoods from df_chamados
chamados_bairros = set(df_chamados['bairro'].dropna().unique())

# Get all unique neighborhoods from the bacias_bairros dictionary
bacias_known_bairros = set()
for bacia, bairros_list in bacias_bairros.items():
    bacias_known_bairros.update(bairros_list)

# Find bairros in df_chamados that are not in bacias_bairros
bairros_not_in_bacias = chamados_bairros - bacias_known_bairros

if bairros_not_in_bacias:
    print("Os seguintes bairros existem em 'df_chamados' mas não foram encontrados em nenhuma bacia do dicionário 'bacias_bairros':")
    for bairro in sorted(list(bairros_not_in_bacias)):
        print(f"- {bairro}")
else:
    print("Todos os bairros em 'df_chamados' foram encontrados em alguma bacia do dicionário 'bacias_bairros'.")

Os seguintes bairros existem em 'df_chamados' mas não foram encontrados em nenhuma bacia do dicionário 'bacias_bairros':
- ACAMPAMENTO ANCHIETA
- BAIRRO INEXISTENTE
- CAMPO GRANDE
- CD SAO JORGE
- ESTANCIA DO RIO GRANDE
- JD CLUBE DE CAMP
- JD CLUBE DE CAMPO
- JD DO MIRANTE
- JD ESPIRITO SANTO
- JD GUARIPOCABA
- JD JOAQUIM EUGENIO D
- JD RIVIERA
- JD SANTO ANTONIO DE
- JD SILVANA
- JD SILVIA
- PARANAPIACABA
- PQ AMERICA
- PQ DAS GARCAS
- PQ DO PEDROSO
- PQ MIAMI
- PQ REPRESA BILLINGS
- PQ RIO GRANDE
- RECREIO DA BORDA DO CAMPO
- RIO GRANDE
- SITIO TAQUARAL
